# Notebook 06: ReLU and why nonlinearity matters

In the last notebook, a neuron made a linear value: multiply inputs by weights, add the pieces, then add a bias.
This notebook asks what happens when we put another linear layer after that.
First, we will see that linear layer plus linear layer can still be rewritten as one linear calculation.
Then we add ReLU, a small rule that creates a bend by shutting off negative hidden values.

By the end, you should be able to explain ReLU in two directions:

- forward pass: which hidden values reach the next layer
- backward pass: which gradients are allowed to flow back


### Why two linear layers can collapse into one

Before ReLU, look at the case with no activation function between layers.
An activation function is an extra rule applied after a layer; in this section, there is no extra rule yet.

The `hidden` value is the intermediate result from the first linear layer.
The `output` value is the final result from the second linear layer, which uses `hidden` as its input.

These two lines are like two tiny `nn.Linear` layers with one input and one output each:

```python
hidden = 3 * x + 1
output = 4 * hidden - 5
```

The second layer does **not** add `hidden` and `output` together.
It uses the already-computed `hidden` value to make `output`.

To collapse the chain, replace `hidden` with the formula that created it:

```text
output = 4 * hidden - 5
output = 4 * (3 * x + 1) - 5
output = 12 * x + 4 - 5
output = 12 * x - 1
```

So for `x = 2`, both paths produce `23`.
The word `collapsed` means that the two-step chain has been rewritten as one direct formula from `x` to `output`.

Important lesson: adding a hidden linear layer does not automatically make the model more powerful.
We need a nonlinearity between linear layers so the whole stack cannot flatten back into one straight-line formula.


In [20]:
x = 2

hidden = 3 * x + 1  # like first Linear layer
output = 4 * hidden - 5  # like second Linear layer

collapsed_output = 12 * x - 1  # the final output from the two-layer linear chain, rewritten as one single linear formula
print(output)
print(collapsed_output)

23
23


### ReLU as a forward gate

ReLU stands for Rectified Linear Unit.
For this course, you can think of it as a gate after a hidden linear value.

The rule is:

```text
relu(x) = max(0.0, x)
```

The `max` function picks the larger of two numbers.
That gives ReLU three simple cases:

- if `x < 0.0`, ReLU returns `0.0`
- if `x == 0.0`, ReLU returns `0.0`
- if `x > 0.0`, ReLU returns `x`

This is why the gate metaphor works: negative values are closed off, and positive values pass through unchanged.
The next code cell tests one negative input, zero, and one positive input so we can see the whole rule.


In [21]:
def relu(x: float) -> float:
    return max(0.0, x)

for value in [-2.0, 0.0, 3.0]:
    print(value, "->", relu(value))

-2.0 -> 0.0
0.0 -> 0.0
3.0 -> 3.0


### ReLU derivative as a backward gate

A derivative tells us the local slope: how much the output changes when the input moves a tiny amount.
When `x > 0.0`, ReLU returns `x`, so the slope is `1.0`.
When `x < 0.0`, ReLU stays flat at `0.0`, so the slope is `0.0`.
At exactly `x == 0.0`, ReLU has a corner; for this course, we use `0.0` as the derivative there.

Backward meaning: `1.0` lets the incoming gradient pass through, and `0.0` stops it.


In [22]:
def relu_derivative(x: float) -> float:
    return 1.0 if x > 0.0 else 0.0

In [23]:
for value in [-2.0, 0.0, 3.0]:
    print(value, "->", relu_derivative(value))

-2.0 -> 0.0
0.0 -> 0.0
3.0 -> 1.0


### How ReLU changes backward flow

During backpropagation, a later layer sends a gradient back to this hidden value.
ReLU multiplies that incoming gradient by its derivative.

If the ReLU gate was open, the derivative is `1.0`, so the gradient passes through.
If the ReLU gate was closed, the derivative is `0.0`, so the gradient becomes `0.0`.

In [24]:
gradient_from_later_layer = 5.0

for hidden_before_relu in [-2.0, 3.0]:
    gradient_to_earlier_layer = (
        gradient_from_later_layer * relu_derivative(hidden_before_relu)
    )
    print(hidden_before_relu, "->", gradient_to_earlier_layer)

-2.0 -> 0.0
3.0 -> 5.0


### Checkpoint: compute ReLU by hand

Use:

```text
gradient_from_later_layer = 5.0
```

Fill in the table before running more code.

| hidden_before_relu | ReLU output | ReLU derivative | gradient_to_earlier_layer |
|---:|---:|---:|---:|
| -2.0 | 0.0 | 0.0 | 0.0 |
| 0.0 | 0.0 | 0.0 | 0.0 |
| 3.0 | 3.0 | 1.0 | 5.0 |

Remember: `gradient_to_earlier_layer = gradient_from_later_layer * relu_derivative(hidden_before_relu)`.


### How several ReLUs combine in one hidden layer

A hidden neuron is just one output slot from a layer.
It is not a physical object or a special Python object.
For one input number `x`, one hidden output slot has one weight and one bias:

```text
hidden_before_relu = weight * x + bias
hidden_after_relu = relu(hidden_before_relu)
```

A wider hidden layer has several output slots in the same forward pass.
For example, a layer with one input and three hidden outputs is like `nn.Linear(in_features=1, out_features=3)`.
That means the same `x` is sent to three different learned formulas:

```text
z1 = weight_1 * x + bias_1
z2 = weight_2 * x + bias_2
z3 = weight_3 * x + bias_3

h1 = relu(z1)
h2 = relu(z2)
h3 = relu(z3)
```

Those three ReLU values are in the same hidden layer, not three separate layers.
The output layer then combines them with another weighted sum:

```text
output = output_weight_1 * h1 + output_weight_2 * h2 + output_weight_3 * h3 + output_bias
```

This is how several ReLU bends can make one nonlinear shape.
The hidden weights and biases decide where each ReLU turns on.
The output weights decide whether each ReLU ramp pushes the final output up or down.


In [25]:
hidden_weights: list[float] = [1.0, 1.0, 1.0]
hidden_biases: list[float] = [0.0, -2.0, -4.0]
output_weights: list[float] = [1.0, -2.0, 1.0]
output_bias = 0.0


def hidden_relu_values(x: float) -> list[float]:
    hidden_before_relu = [
        weight * x + bias
        for weight, bias in zip(hidden_weights, hidden_biases)
    ]
    return [relu(value) for value in hidden_before_relu]


def model_output(x: float) -> float:
    hidden_after_relu = hidden_relu_values(x)
    return (
        sum(
            output_weight * hidden_value
            for output_weight, hidden_value in zip(output_weights, hidden_after_relu)
        )
        + output_bias
    )


print("x -> hidden_after_relu -> output")
for x in [-1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0]:
    print(x, "->", hidden_relu_values(x), "->", model_output(x))


x -> hidden_after_relu -> output
-1.0 -> [0.0, 0.0, 0.0] -> 0.0
0.0 -> [0.0, 0.0, 0.0] -> 0.0
1.0 -> [1.0, 0.0, 0.0] -> 1.0
2.0 -> [2.0, 0.0, 0.0] -> 2.0
3.0 -> [3.0, 1.0, 0.0] -> 1.0
4.0 -> [4.0, 2.0, 0.0] -> 0.0
5.0 -> [5.0, 3.0, 1.0] -> 0.0
